In [ ]:
# 실습에 필요한 패키지 설치 (최초 1회 실행)
!pip install -q pandas tokenizers


# 1. 토큰화 (Tokenization)

> **📌 이 노트북의 위치**
>
> 교안 실습(2_0)에서는 `AutoTokenizer`로 **이미 만들어진** 토크나이저를 사용했다.
> 이 노트북에서는 **토크나이저를 직접 학습**시켜 본다.
> "단어 사전은 어떻게 만들어지는가?"에 대한 답을 직접 구현하며 이해하는 것이 목표이다.

- 텍스트를 숫자로 변환하는 과정
- 딥러닝 모델은 `안녕하세요`와 같은 문자를 직접 이해할 수 없다
- 오직 숫자만을 입력으로 받을 수 있으므로, 텍스트를 모델이 이해할 수 있는 숫자 형태로 변환해야 한다
- **토큰 ID 시퀀스**로 데이터를 변환하는 역할을 **토크나이저**가 수행한다

## 1-1. 데이터 준비: 말뭉치(Corpus) 구축

- 토크나이저를 생성하기 위해서 가장 먼저 고려해야 할 것은 `어떤 단어를 어떻게 나눌 것인가?`에 대한 기준을 학습시키는 것
- 학습이 필요하다 → 학습할 데이터가 필요하다

### 1-1-1. 데이터 다운로드

- `!wget [URL]`: URL이 가리키는 파일을 현재 작업 디렉토리에 다운로드
- 네이버 영화 리뷰 데이터셋(NSMC)을 사용한다


In [ ]:
import urllib.request
import os

# 데이터 다운로드 (wget 대신 Python 표준 라이브러리 사용)
# → Windows 환경에서는 wget이 설치되어 있지 않을 수 있으므로
#   OS에 관계없이 동작하는 urllib을 사용한다.
#
# 주소는 raw.githubusercontent.com을 직접 사용한다.
# github.com/.../raw/... 형태는 내부적으로 한 번 더 리다이렉트되므로 실패 확률이 조금 더 높다.
url = 'https://raw.githubusercontent.com/e9t/nsmc/master/ratings.txt'
filename = 'ratings.txt'

if not os.path.exists(filename):
    print(f'{filename} 다운로드 중... (약 20MB, 네트워크에 따라 수십 초 소요)')
    urllib.request.urlretrieve(url, filename)
    print(f'{filename} 다운로드 완료!')
else:
    print(f'{filename} 이미 존재합니다. 다운로드를 건너뜁니다.')

print(f'파일 크기: {os.path.getsize(filename) / 1024 / 1024:.1f} MB')


### 1-1-2. 데이터 정제 및 학습용 파일 생성

1. 데이터 정제
    - `ratings.txt` 원본 파일에는 **리뷰 텍스트** 외에 ID, 레이블 등 불필요한 정보나 비어있는 행 (결측치)도 포함되어 있음.
    - 이러한 데이터들의 경우 모델 학습에 방해되므로 전처리 과정이 필요
    1. pandas를 활용하여 결측치 제거
    2. 리뷰 내용이 담긴 열만 추출
    3. 추출한 텍스트를 `줄바꿈`으로 연결하여 새로운 텍스트 파일로 저장 (원본 보장)

In [ ]:
import pandas as pd

# 데이터 불러오기
# sep='\t': ratings.txt는 탭(Tab)으로 열이 구분된 파일이다.
data = pd.read_csv('ratings.txt', sep='\t')
# 데이터 정보 확인
print(data.info())  # id, document, label
print(data.head())  # 상위 5개 데이터 확인

# 결측치 제거
# 리뷰 내용이 비어 있는 행이 8개 정도 존재한다. 그대로 두면 학습 시 오류가 난다.
data = data.dropna()
# 리뷰 내용이 담긴 열만 추출: 'document'
data = data['document']
print(f'\n정제 후 리뷰 개수: {len(data):,}')

# ========== 추출한 텍스트를 파일로 저장 ==========
#
# ⚠️ 여기서 to_csv()를 쓰면 안 된다.
#    data.to_csv('nsmc.txt', index=False, header=False, sep='\n')
#    → to_csv는 CSV 규격을 지키느라, 쉼표나 따옴표가 들어간 리뷰를
#      큰따옴표로 감싸고 내부 따옴표를 두 번씩 반복해서 저장한다.
#
#      원본: 우리애기첫영화 너무너무 잘선택한거같아요""ㅎ굳
#      저장: "우리애기첫영화 너무너무 잘선택한거같아요""""ㅎ굳"     ← 원문 훼손!
#
#      실제로 약 850줄이 이런 식으로 오염된다. 토크나이저는 이 따옴표까지
#      '진짜 데이터'로 알고 학습하게 되므로, 단어 사전에 쓰레기 토큰이 섞인다.
#
# ⭕ 우리가 원하는 것은 "한 줄에 리뷰 하나"인 순수 텍스트 파일이다.
#    그러면 CSV 함수를 쓸 이유가 없다. 그냥 줄바꿈으로 이어 붙여 저장하면 된다.
with open('nsmc.txt', 'w', encoding='utf-8') as f:
    f.write('\n'.join(data))

print('nsmc.txt 저장 완료')


2. 추출한 데이터 정보 확인

In [ ]:
# 추출한 데이터 정보 확인
with open('nsmc.txt', 'r', encoding='utf-8') as file:
    lines = file.readlines()
    print(f"총 리뷰 개수: {len(lines):,}")
    print("샘플 리뷰:")
    for line in lines[:5]:  # 상위 5개 리뷰 출력
        print(line.strip()) # 앞뒤 공백 제거

# ========== 저장이 제대로 되었는지 검증 ==========
# 위에서 정제한 개수와 파일의 줄 수가 같아야 한다.
assert len(lines) == len(data), f'줄 수가 다릅니다! 원본 {len(data)} vs 파일 {len(lines)}'

# 원문에 없던 따옴표가 새로 생기지 않았는지 확인
quoted = [l for l in lines if l.startswith('"') and l.rstrip().endswith('"')]
print(f'\n앞뒤가 따옴표로 감싸진 줄: {len(quoted)}개')
print('→ 0에 가까우면 정상. to_csv로 저장했다면 수백 개가 나온다.')
print('✅ 검증 통과: 원문이 훼손 없이 저장되었습니다.')


## 1-2. 토크나이저 학습 및 활용

- 준비된 말뭉치(`nsmc.txt`)를 바탕으로, 문장을 어떤 규칙에 따라 토큰으로 분해 할지 정의
- **단어 사전(Vocabulary)를 구축**

### 1-2-1. 토크나이저(Tokenizer) 종류

1. 단어/규칙 기반
    - 가장 직관적이고 간단한 방식. 딥러닝 모델 등장 전부터 사용
    
    | 종류 | 설명 | 특징 |
    | --- | --- | --- |
    | 공백 기반 토크나이저 | 단순히 `공백` 기준으로 텍스트를 단어 단위로 나눔 예: `I love apples` → [’I’, ‘love’, ‘apples’] | 가장 단순하고 빠름. 단점: 한국어와 같이 단어 경계가 명확하지 않은 언어는 적용하기 어려움. `저는 개발자가 되고 싶어요.’ → 저는? 개발자? 개발자가? 되다. 싶어요. 등… 경계가 모호 |
    | 형태소 분석기 | 단어를 의미의 최소 단위인 형태소로 분리 (주로 한국어에 사용) 예: `공부하고` → [’공부’, ‘+하’, ‘+고’] | 한국어 처리에 특히 강함. 단점: 특정 언어에 의존적. 형태소 분석기 성능에 따라 품질이 달라짐. |
2. 서브워드 기반
    - 현대 언어 모델(LLM)에서 가장 널리 사용되는 토큰화 방식
    - 단어와 문자 수준의 장점을 결합하여 **OOV 문제를 최소화**하고 어휘 크기를 효율적으로 관리
    - **OOV(Out-Of-Vocabulary)**
        - 전처리 된 단어 목록에 존재하지 않는 새로운 단어가 등장하는 상황
        - 신조어, 오타, 희귀 단어, 형태 변화 등에 의해 발생 할 수 있음.
        - ex) 공백 기반 토크나이저로 처리한 데이터에 `나` 에 대한 문자가 다음과 같이 등장 할 수 있음.
            - 나는, 나를, 나에게, 나의, 난 등.
        - ex) 하지만, 실제 문장에서는 `나`에 대한 문자 중, 다음과 같은 상황도 있을 수 있음.
            - 나만, 나, 나에 등
        - 이러한 경우, 사전에 없는 단어가 등장하여 학습하지 않은 단어처럼 처리될 수 있음.
    
    | 종류 | 설명 | 특징 |
    | --- | --- | --- |
    | BPE (Byte Pair Encoding) | 가장 빈번하게 등장하는 문자 쌍을 찾아 새로운 토큰으로 병합 | 간단하고 효율적. GPT-3/4등 OpenAI 계열 모델에서 주로 사용 |
    | **WordPiece** | 통계적 확률을 기반으로 가장 가능성이 높은 하위 문자열 쌍을 병함 | 확률 기반으로, BPE보다 언어 가능성을 최대화. BERT등 구글 및 관련 계열의 모델에서 주로 사용 |

### 1-2-2. WordPiece 알고리즘과 토크나이저 초기화

- `BertWordPieceTokenizer`: BERT에서 사용된 WordPiece 방식의 토크나이저
    - BERT: 구글이 발표한 사전 훈련 언어 모델
    - 사전에 없는 새로운 단어**(Out-of-Vocabulary, OOV)**가 등장했을 때, 가능하다면, 아는 단어들의 조합으로 분해함.
        - ex) `자연어처리` 단어가 사전에 없지만, ‘자연어’와 ‘처리’ 는 사전에 있다면, **[‘자연어’, ‘##처리’]** 형태로 분해하여 처리함.
        - ex) 만약, 이렇게 분해조차 할 수 없다면 `[UNK]`(알 수 없음) 토큰으로 처리.
    - `##`: 앞 토큰에 이어서 붙인다는 의미의 접두사.

In [ ]:
from tokenizers import BertWordPieceTokenizer

# 아직 학습되지 않은, 비어있는 토크나이저 객체 생성
# BertWordPieceTokenizer: BERT에서 사용된 WordPiece 방식의 토크나이저
#
# 교안 실습(2_0)에서는 AutoTokenizer로 "이미 학습된" 토크나이저를 불러왔다.
# 여기서는 빈 토크나이저를 만들고, 아래에서 직접 데이터로 학습시킬 것이다.
#
# lowercase=False: 대소문자를 구분 (한국어에는 대소문자가 없지만 영어 혼합 대비)
# strip_accents=False: 악센트 기호를 유지
tokenizer = BertWordPieceTokenizer(lowercase=False, strip_accents=False)


### 1-2-3. 토크나이저 학습

- `.train()`: 말뭉치 파일을 분석해서, 설정된 `vocab_size`에 맞춰 가장 효율적인 단어 사전을 생성
- `special_tokens`: 모델이 문장의 구조를 이해하거나 특정 과제를 수행하기 위해 사용하는 **특수 목적의 토큰**
    1. `[CLS]`(Classification): 문장의 시작을 의미. 문장 전체 정보를 요약하는 역할
    2. `[SEP]` (Separator): 두 문장을 구분
    3. `[PAD]` (Padding): 여러 문장을 한번에 처리(배치 처리)할 때, 길이를 맞춰주기 위해 짧은 문장 뒤에 채워 넣음.
        - ex) 배치 사이즈가 2일 때, 2개의 데이터. 
        [’안녕’, ‘하세요.’] 문장과 [’저는’, ‘개발자’, ‘입니다.’] 문장을 한 번에 처리한다면,
        [’안녕’, ‘하세요.’, ‘[PAD]’]  와 같이 처리
    4. `[UNK]` (Unknown): 사전에 없으면서 WordPiece로도 분해할 수 없는 단어를 위해 사용.
    5. `[MASK]` (Masking): BERT의 사전 학습(Pre-training) 시, 단어를 가리는 용도로 사용.

In [ ]:
# 토크나이저 학습
# vocab_size: 토큰 사전의 크기
    # 이 크기는 성능과 효율성 사이의 균형을 맞추는 데 중요
    # 작으면 → 단어가 잘게 쪼개져 문장당 토큰 수가 늘어남
    # 크면   → 임베딩 표가 커져 메모리를 많이 씀
# min_frequency: 단어가 토큰 사전에 포함되기 위한 최소 빈도 수
    # 2로 두면 딱 한 번만 등장한 오타나 희귀 표현은 사전에 넣지 않는다
# special_tokens: 모델에서 특별한 의미를 가지는 토큰들
    # 리스트의 순서대로 0, 1, 2, 3, 4번 ID를 배정받는다
#
# 20만 문장을 분석하므로 30초~2분 정도 걸린다. 진행 막대가 멈춘 것처럼 보여도 기다리자.
tokenizer.train(
    files='nsmc.txt',
    vocab_size=30000,
    min_frequency=2,
    special_tokens=["[PAD]", "[UNK]", "[CLS]", "[SEP]", "[MASK]"],
)

# ========== 학습 결과를 파일로 저장 ==========
# 저장하지 않으면 노트북을 다시 열 때마다 20만 문장을 다시 학습해야 한다.
# save_model()은 단어 사전을 'nsmc-vocab.txt' 파일로 남긴다.
#   → 2_2, 2_3 노트북에서 이 파일을 불러와 재학습 없이 바로 사용할 것이다.
saved = tokenizer.save_model('.', 'nsmc')
print('저장된 파일:', saved)


In [ ]:
# 토크나이저의 어휘 사전 크기 확인
print(f"어휘 사전 크기: {tokenizer.get_vocab_size()}")
# 참고: 목표로 지정한 30,000에 못 미칠 수도 있다.
#       min_frequency 조건을 통과한 후보가 부족하면 목표치를 채우지 못하기 때문이다.

# 토크나이저의 어휘 사전 무작위 10개 단어 확인
# 재현성 위해 시드 고정
import random
random.seed(42)

vocab = tokenizer.get_vocab()
sorted_vocab = sorted(vocab.items(), key=lambda x: x[1])  # 인덱스 기준으로 정렬
print("\n어휘 사전 무작위 10개 단어:")
for token, index in random.sample(sorted_vocab, 10):
    print(f"{index}: {token}")

# ========== 사전이 어떻게 구성되어 있는지 관찰 ==========
print("\n[앞쪽 20개] 특수 토큰과 기본 글자들")
print([t for t, i in sorted_vocab[:20]])

print("\n[뒤쪽 15개] 병합을 많이 거친 긴 토큰들")
print([t for t, i in sorted_vocab[-15:]])

# 앞쪽은 한 글자, 뒤로 갈수록 긴 덩어리가 나온다.
# WordPiece는 '글자'에서 출발해 자주 붙어 다니는 조각을 합쳐 나가기 때문이다.


### 1-2-4. 토큰화 실행 및 결과 확인

- 학습이 완료된 토크나이저는 어떤 문장이든 토큰 시퀀스와 ID 시퀀스로 변환 가능
- `.encode`: 사용자가 입력한 텍스트를 BERT 모델이 이해하고 처리할 수 있는 숫자 형태로 변환
    1. **토큰화:** 입력 문장을 토큰 시퀀스로 분할 (WordPiece 규칙 적용)
    2. **정수 인코딩:** 각 토큰을 미리 학습된 사전에 있는 고유한 정수 ID로 변환

In [ ]:
text = '나는 개발자 입니다!'
encoded = tokenizer.encode(text)

# .tokens: 분리된 토큰들의 리스트
print('토큰화 결과:', encoded.tokens)
# 출력 예: ['나는', '개발', '##자', '입니다', '!']
# → '개발자'가 사전에 통째로 없으면 '개발' + '##자'로 분해된다 (WordPiece)
# → '##'은 "앞 토큰에 이어 붙인다"는 의미 (교안 슬라이드 15, 17)

# .ids: 각 토큰에 매핑된 고유 정수 ID의 리스트
print('정수 인코딩:', encoded.ids)
# ⚠️ ID 값은 교재와 다르게 나오는 것이 정상이다.
#    학습 데이터와 설정에 따라 사전이 만들어지는 순서가 달라지기 때문이다.
#    중요한 것은 '어떤 숫자가 나왔는가'가 아니라 '토큰이 숫자로 바뀌었는가'이다.

# 디코딩: 숫자를 다시 문장으로 되돌릴 수 있는지 확인
print('디코딩 결과:', tokenizer.decode(encoded.ids))

# 이 정수 ID가 교안 실습(2_0)에서 본 input_ids와 같은 역할을 한다.
# 차이점: 여기서는 직접 학습한 토크나이저를 사용했고,
#         2_0에서는 이미 학습된 klue/bert-base 토크나이저를 사용했다.


> **🤔 2_0과 다른 점 하나 — `[CLS]` 와 `[SEP]` 이 안 붙었다!**
>
> 2_0에서 `AutoTokenizer`로 인코딩했을 때는 문장 앞뒤에 `[CLS]`, `[SEP]`이 자동으로 붙었다.
> 그런데 지금은 붙지 않는다. 왜일까?
>
> 특수 토큰을 붙이는 규칙을 **후처리기(post-processor)** 라고 하는데,
> **빈 토크나이저를 만들어 `.train()`만 한 경우에는 이 후처리기가 설정되지 않는다.**
> 사전(vocab)은 만들어졌지만 "앞뒤에 뭘 붙여라"는 규칙은 비어 있는 상태인 것이다.
>
> **해결 방법: 저장한 사전 파일에서 다시 불러오면 된다.**
>
> ```python
> from tokenizers import BertWordPieceTokenizer
>
> # 방금 저장한 nsmc-vocab.txt를 넣어서 생성
> tok2 = BertWordPieceTokenizer('nsmc-vocab.txt', lowercase=False, strip_accents=False)
> print(tok2.encode('나는 개발자 입니다!').tokens)
> # → ['[CLS]', '나는', '개발', '##자', '입니다', '!', '[SEP]']  ✅ 붙는다!
> ```
>
> 사전 파일을 주고 생성하면 라이브러리가 BERT 규격의 후처리기를 함께 설정해 주기 때문이다.
> 👉 **그래서 2_2, 2_3에서는 재학습하지 않고 이 파일을 불러오는 방식을 쓴다.** 빠르고, 동작도 2_0과 같아진다.

> **🧪 `[UNK]` 는 언제 나오는가 — 직접 확인해 보기**
>
> ```python
> tests = ['나는 개발자 입니다', '싸피에서 AI를 배운다', '쒧뷁훀 영화', '👍 최고']
> for s in tests:
>     enc = tokenizer.encode(s)
>     n_unk = enc.tokens.count('[UNK]')
>     print(f'{s:22s} UNK {n_unk}개 -> {enc.tokens}')
> ```
>
> - 우리 사전은 **네이버 영화 리뷰**만 보고 만들어졌다.
> - 영화 리뷰에 거의 안 나오는 글자(희귀 한자, 특수문자, 이모지)는 `[UNK]`가 된다.
> - 👉 **토크나이저는 학습한 데이터를 닮는다.** 의료 문서를 다루려면 의료 말뭉치로 학습해야 한다.


In [ ]:
tests = ['나는 개발자 입니다', '싸피에서 AI를 배운다', '쒧뷁훀 영화', '👍 최고']
for s in tests:
    enc = tokenizer.encode(s)
    n_unk = enc.tokens.count('[UNK]')
    print(f'{s:22s} UNK {n_unk}개 -> {enc.tokens}')